In [4]:
# papermill parameters

garch_lookback=1000
garch_dist='skewt'
input_file='usdt.parquet'
output_file='garch-forecast.parquet'
jobs_concurrency=20

In [5]:
print(f'''
GJR-GARCH volatility forecast evaluation
========================================

input_file={input_file}
output_file={output_file}

garch_lookback={garch_lookback}
garch_dist={garch_dist}

jobs_concurrency={jobs_concurrency}
''')


GJR-GARCH volatility forecast evaluation

input_file=usdt.parquet
output_file=garch-forecast.parquet

garch_lookback=1000
garch_dist=skewt

jobs_concurrency=20



Fit GJR-GARCH model and forecast
================================

In [7]:
import polars as pl
from arch import arch_model
import numpy as np
from tqdm import tqdm

df = pl.read_parquet(input_file).sort([
    'symbol', 'ts'
]).filter(
    (pl.col('open') > 0) & (pl.col('high') > 0) &
    (pl.col('low') > 0) & (pl.col('close') > 0) &
    (pl.col('base_volume') > 0)
).select([
    pl.col('ts'), pl.col('symbol'),
    # todays log return
    (pl.col('close') / pl.col('close').shift(1)).log().over('symbol').alias('ret'),
])

from numpy.lib.stride_tricks import sliding_window_view
from tqdm.auto import tqdm
from joblib import Parallel, delayed

# sigma, mu, omega, alpha, gamma, beta, eta, lambda
def forecast_sigma(returns: np.ndarray, symbol: str) -> np.ndarray:
    def _fc(returns: np.ndarray) -> np.ndarray:
        try:
            gjr = arch_model(
                returns * 100,
                vol='Garch',
                p=1,
                o=1,
                q=1,
                dist=garch_dist,
                rescale=False
            ).fit(
                disp='off',
                show_warning=False,
            )

            return np.array([
                np.sqrt(gjr.forecast(horizon=1).variance.values[-1, 0]) / 100,
                gjr.params.get('mu', np.nan),
                gjr.params.get('omega', np.nan),
                gjr.params.get('alpha[1]', np.nan),
                gjr.params.get('gamma[1]', np.nan),
                gjr.params.get('beta[1]', np.nan),
                gjr.params.get('nu', np.nan),
                gjr.params.get('lambda', np.nan),
            ])
            
        except Exception as e:
            print(f'''caught {e}, returning NaN''')
            return np.full(8, np.nan)

    ws = sliding_window_view(returns, garch_lookback)
    fc = [_fc(w) for w in ws] # len(returns) - garch_lookback + 1
    pad = np.full((garch_lookback, 8), np.nan) # garch_lookback
    r = np.vstack([pad, *fc[:-1]]) # garch_lookback + len(returns) - garch_lookback + 1 - 1 = len(returns)
    
    if len(r) != len(returns):
        print(f"ERROR: {symbol} size mismatch! Returns: {len(returns)}, Result: {len(r)}")
    
    return r



symbols = (
    df.group_by('symbol', maintain_order=True)
        .agg([pl.col('ret'),pl.col('ts')])
        .filter(pl.col('ret').list.len() >= garch_lookback))

res = Parallel(n_jobs=jobs_concurrency)(
    tqdm([
        delayed(forecast_sigma)(np.array(s['ret']), s['symbol'])
        for s in symbols.iter_rows(named=True)
    ])
)

schema = {
    'forecast': pl.Float64,
    'mu': pl.Float64,
    'omega': pl.Float64,
    'alpha[1]': pl.Float64,
    'gamma[1]': pl.Float64,
    'beta[1]': pl.Float64,
    'nu': pl.Float64,
    'lambda': pl.Float64,
}

pl.concat([
    pl.DataFrame(r, schema=schema).with_columns([
        pl.lit(symbols['symbol'][i]).alias('symbol'),
        pl.Series('ts', symbols['ts'][i])
    ]) for i, r in enumerate(res)
]).write_parquet(output_file)

  0%|          | 0/337 [00:00<?, ?it/s]

KeyboardInterrupt: 

Evaluate model against daily R/S volatility
===========================================

In [ ]:
%%markdown

res = (
    df.join(pl.read_parquet(output_file), on=['ts','symbol'])
        .drop_nulls()
        .sort(['symbol','ts'])
        .group_by('symbol')
        .agg([
            ((pl.col("sigma_gjr") - pl.col("sigma_rs"))**2).mean().sqrt().alias("rmse"),
            (pl.col("sigma_gjr") - pl.col("sigma_rs")).abs().mean().alias("mae"),    
            (pl.col("sigma_gjr") - pl.col("sigma_rs")).mean().alias("bias")
        ])
    .select([pl.col('rmse'), pl.col('mae'), pl.col('bias')]).describe())

import matplotlib.pyplot as plt

# Metrics to plot
metrics = ["rmse", "mae", "bias"]
colors = ["#3498db", "#e74c3c", "#2ecc71"]

plt.figure(figsize=(18, 6))

# Create a layout with 3 subplots
for i, metric in enumerate(metrics):
    plt.subplot(1, 3, i + 1)
    
    # We use .to_numpy() to ensure compatibility with matplotlib
    plt.hist(res[metric].to_numpy(), bins=20, color=colors[i], edgecolor='black', alpha=0.7)
    
    plt.title(f'Distribution of {metric.upper()}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()